# Lab 2 — Build a Tiny Generation Loop

## Week 1: Foundations of LLM Inference

This notebook demystifies how language models generate text.

The central idea:

> A language model does not generate a whole answer at once.  
> It repeatedly predicts the next token.

By the end of this notebook, you should understand:
- how a prompt becomes token IDs
- how the model performs a forward pass
- what logits are
- how greedy decoding works
- why generation is sequential
- why decoding becomes a production bottleneck
- why KV cache, batching, and speculative decoding exist


## 1. Why This Lab Matters

Most people experience LLMs through chat interfaces.

They type:

```text
Explain reinforcement learning.
```

and a response appears.

But underneath the interface, the model is doing something much more mechanical:

```text
Prompt → Forward Pass → Logits → Choose Token → Append Token → Repeat
```

This loop is the beating heart of LLM inference.


## 2. Install Dependencies

Run this in Google Colab or a fresh local environment.

If you already installed the lab requirements, skip this cell.


In [ ]:
# Uncomment if needed
# !pip install transformers torch accelerate pandas matplotlib tabulate

## 3. Import Libraries

In [ ]:
import time
import torch
import pandas as pd
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Choose a Small Model

We use:

```text
Qwen/Qwen2.5-0.5B-Instruct
```

This lab is about mechanics, not model quality.


In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

## 5. Load Tokenizer and Model

The tokenizer converts text into token IDs.

The model performs next-token prediction.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print("Loaded tokenizer and model.")
print("Vocabulary size:", tokenizer.vocab_size)

## 6. Start With a Prompt

In [ ]:
prompt = "The capital of Ghana is"

encoded = tokenizer(prompt, return_tensors="pt")
input_ids = encoded["input_ids"].to(device)

print("Prompt:", prompt)
print("Input IDs:", input_ids)
print("Input shape:", input_ids.shape)
print("Number of input tokens:", input_ids.shape[-1])

## 7. Decode the Input IDs

Token IDs are just integers.

Let’s convert them back to text and token pieces.


In [ ]:
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

rows = []
for i, (token_id, token) in enumerate(zip(input_ids[0].tolist(), tokens)):
    rows.append({
        "position": i,
        "token_id": token_id,
        "token_piece": token,
    })

pd.DataFrame(rows)

## 8. Run One Forward Pass

This is the most important line in the notebook:

```python
outputs = model(input_ids=input_ids)
```

This single call runs the transformer forward pass.


In [ ]:
with torch.no_grad():
    outputs = model(input_ids=input_ids)

logits = outputs.logits

print("Logits shape:", logits.shape)

## 9. Understand the Logits Shape

The shape is:

```text
(batch_size, sequence_length, vocabulary_size)
```


In [ ]:
batch_size, sequence_length, vocab_size = logits.shape

shape_info = pd.DataFrame([
    {"dimension": "batch_size", "value": batch_size, "meaning": "Number of sequences processed together"},
    {"dimension": "sequence_length", "value": sequence_length, "meaning": "Number of tokens in the current input"},
    {"dimension": "vocab_size", "value": vocab_size, "meaning": "Number of possible next tokens scored"},
])

shape_info

## 10. Extract Final-Position Logits

For generation, we care about the logits at the final token position.


In [ ]:
next_token_logits = logits[:, -1, :]
print("Next-token logits shape:", next_token_logits.shape)

## 11. Inspect Top Candidate Tokens

The model produces a probability distribution over possible next tokens.


In [ ]:
def top_k_predictions(tokenizer, next_token_logits, k=10):
    probabilities = torch.softmax(next_token_logits, dim=-1)
    values, indices = torch.topk(probabilities, k=k)

    rows = []
    for rank, (prob, idx) in enumerate(zip(values[0], indices[0]), start=1):
        token_id = idx.item()
        token_text = tokenizer.decode([token_id])
        rows.append({
            "rank": rank,
            "token_id": token_id,
            "token_text": repr(token_text),
            "probability": round(prob.item(), 6),
        })

    return pd.DataFrame(rows)

top_k_predictions(tokenizer, next_token_logits, k=10)

## 12. Greedy Decoding

Greedy decoding chooses the token with the highest score/probability.


In [ ]:
next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)

print("Selected token ID:", next_token_id.item())
print("Selected token text:", repr(tokenizer.decode(next_token_id[0])))

## 13. Append the Generated Token

Autoregressive generation means the newly generated token becomes part of the next input.


In [ ]:
new_input_ids = torch.cat([input_ids, next_token_id], dim=-1)

print("Original sequence length:", input_ids.shape[-1])
print("New sequence length:", new_input_ids.shape[-1])
print("Decoded text:", tokenizer.decode(new_input_ids[0], skip_special_tokens=True))

## 14. Build the Generation Loop

Now we repeat:

```text
forward pass → final logits → choose token → append token → repeat
```


In [ ]:
def greedy_generate_step(model, tokenizer, input_ids):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        logits = outputs.logits
        next_token_logits = logits[:, -1, :]
        next_token_id = torch.argmax(next_token_logits, dim=-1, keepdim=True)
        new_input_ids = torch.cat([input_ids, next_token_id], dim=-1)

    return new_input_ids, next_token_id, next_token_logits


prompt = "The capital of Ghana is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

max_new_tokens = 20
history = []

for step in range(max_new_tokens):
    start = time.perf_counter()

    input_ids, next_token_id, next_token_logits = greedy_generate_step(
        model=model,
        tokenizer=tokenizer,
        input_ids=input_ids,
    )

    if device == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()

    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

    history.append({
        "step": step + 1,
        "new_token_id": next_token_id.item(),
        "new_token_text": repr(tokenizer.decode(next_token_id[0])),
        "sequence_length": input_ids.shape[-1],
        "step_latency_sec": end - start,
        "text_so_far": generated_text,
    })

pd.DataFrame(history)[["step", "new_token_id", "new_token_text", "sequence_length", "step_latency_sec"]]

## 15. Inspect the Final Generated Text

In [ ]:
final_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
print(final_text)

## 16. Visualize Sequence Growth

In [ ]:
history_df = pd.DataFrame(history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["step"], history_df["sequence_length"], marker="o")
plt.xlabel("Generation Step")
plt.ylabel("Sequence Length")
plt.title("Sequence Growth During Autoregressive Generation")
plt.grid(True)
plt.show()

## 17. Visualize Per-Token Latency

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["step"], history_df["step_latency_sec"], marker="o")
plt.xlabel("Generation Step")
plt.ylabel("Step Latency (seconds)")
plt.title("Per-Token Generation Latency")
plt.grid(True)
plt.show()

print("Average step latency:", history_df["step_latency_sec"].mean())
print("Approx generated tokens/sec:", 1 / history_df["step_latency_sec"].mean())

## 18. Why This Shows Decode Is Sequential

At each step:

```text
Token N must exist before Token N+1 can be predicted.
```

That dependency chain limits parallelism.


## 19. Compare Different Prompts

In [ ]:
test_prompts = [
    "The capital of Ghana is",
    "Explain inference engineering in simple terms:",
    "In a production LLM server, latency and throughput",
]

results = []

for prompt in test_prompts:
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    initial_tokens = input_ids.shape[-1]

    start_total = time.perf_counter()

    for _ in range(10):
        input_ids, next_token_id, _ = greedy_generate_step(model, tokenizer, input_ids)

    if device == "cuda":
        torch.cuda.synchronize()

    end_total = time.perf_counter()

    generated_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)

    results.append({
        "prompt": prompt,
        "initial_tokens": initial_tokens,
        "generated_tokens": 10,
        "total_latency_sec": round(end_total - start_total, 4),
        "generated_tok_per_sec": round(10 / (end_total - start_total), 2),
        "output_preview": generated_text[:120],
    })

pd.DataFrame(results)

## 20. Optional: Sampling Instead of Greedy Decoding

Greedy decoding always chooses the top token.

Sampling introduces randomness.


In [ ]:
def sample_next_token(next_token_logits, temperature=1.0):
    scaled_logits = next_token_logits / temperature
    probabilities = torch.softmax(scaled_logits, dim=-1)
    sampled_token = torch.multinomial(probabilities, num_samples=1)
    return sampled_token


prompt = "Artificial intelligence will"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

temperature = 0.8
max_new_tokens = 20

for _ in range(max_new_tokens):
    with torch.no_grad():
        outputs = model(input_ids=input_ids)
        next_token_logits = outputs.logits[:, -1, :]
        next_token_id = sample_next_token(next_token_logits, temperature=temperature)
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)

print(tokenizer.decode(input_ids[0], skip_special_tokens=True))

## 21. Engineering Interpretation

For every generated token, the system must:
- run a forward pass
- compute logits
- select a token
- append it to context
- repeat

At production scale, this loop happens millions or billions of times.

This explains why systems like:
- KV cache
- vLLM
- batching
- FlashAttention
- speculative decoding

exist.


## 22. Connection to KV Cache

In this simplified notebook, every step feeds the full growing `input_ids` back into the model.

That is educational, but inefficient.

Production systems avoid recomputing previous attention state by using:

```text
KV cache
```


## 23. Student Exercises

Complete these:

1. Change the prompt and observe generation behavior.
2. Increase `max_new_tokens` and measure latency.
3. Compare greedy decoding with sampling.
4. Try different temperature values.
5. Explain why decode is sequential.
6. Explain why KV cache would help.


## 24. Reflection Questions

Answer in your lab report:

1. What did this lab reveal about how LLMs generate text?
2. Why are logits important?
3. Why does the sequence length grow?
4. Why does every token require another prediction?
5. Why is decode difficult to parallelize?
6. How does this generation loop affect latency?
7. Why does this lab justify KV cache?
8. Why might speculative decoding be useful?


## Final Lesson

The key lesson from this lab:

> Modern language models are repeated next-token prediction engines.

That single fact explains many of the engineering challenges in production LLM inference.
